# ecgaug on Kaggle

Runs one or more configs of the project at a fixed commit and zips the outputs.
Setup and the team workflow are in `docs/KAGGLE.md`. Short version:

1. Settings (right panel): **Internet on**, Accelerator **GPU** if you want one.
2. Add-ons > Secrets: add `GITHUB_TOKEN` (read access to the repo) and enable it for this notebook.
3. Add Input: attach the team's raw-data dataset (optional but much faster).
4. Edit the parameter cell, then Run All.
5. Download `ecgaug_<sha>_<time>.zip` from Output and run `python scripts/ingest_runs.py <zip>` locally.

In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
GIT_REF = "main"                      # branch name or commit SHA; use your variant branch, e.g. "variant/m1-A"
CONFIGS = ["configs/m1/A.yaml"]       # one or more configs, run in order
MODULE_CLI = "ecgaug.experiments.m1"  # the package CLI for this module
EXTRA_ARGS = ["--device", "auto"]     # passed to the CLI; add "--smoke" for a quick plumbing test
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster)
RAW_DATASET_DIR = "/kaggle/input"     # where the attached raw MIT-BIH dataset is; "" downloads from PhysioNet
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
has_raw = RAW_DATASET_DIR and any(Path(RAW_DATASET_DIR).rglob("100.dat"))
cmd = [sys.executable, "scripts/kaggle_run.py", "--cli", MODULE_CLI, "--env", ENV_MODE, "--out", str(WORK),
       "--config", *CONFIGS]
if has_raw:
    cmd += ["--raw-dir", RAW_DATASET_DIR]
cmd += ["--", *EXTRA_ARGS]
code = subprocess.run(cmd, cwd=REPO).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")